# SpaceX Falcon 9 — Machine Learning Prediction
Predict whether the first stage lands (Class) with LR, SVM, Decision Tree and KNN.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
import warnings; warnings.filterwarnings('ignore')
def plot_confusion_matrix(y, y_predict, title):
    cm = confusion_matrix(y, y_predict)
    ax = plt.subplot(); sns.heatmap(cm, annot=True, ax=ax, fmt='d')
    ax.set_xlabel('Predicted labels'); ax.set_ylabel('True labels'); ax.set_title(title)
    ax.xaxis.set_ticklabels(['did not land', 'land']); ax.yaxis.set_ticklabels(['did not land', 'landed']); plt.show()
data = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv")
X = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv")

In [ ]:
Y = data['Class'].to_numpy()
X = preprocessing.StandardScaler().fit(X).transform(X)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
Y_test.shape

In [ ]:
lr = LogisticRegression()
logreg_cv = GridSearchCV(lr, {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}, cv=10).fit(X_train, Y_train)
print('best params', logreg_cv.best_params_, 'CV acc', logreg_cv.best_score_)
print('test acc', logreg_cv.score(X_test, Y_test))
plot_confusion_matrix(Y_test, logreg_cv.predict(X_test), 'Logistic Regression')

In [ ]:
svm_cv = GridSearchCV(SVC(), {'kernel': ('linear', 'rbf', 'poly', 'sigmoid'), 'C': np.logspace(-3, 3, 5),
                              'gamma': np.logspace(-3, 3, 5)}, cv=10).fit(X_train, Y_train)
print('best params', svm_cv.best_params_, 'CV acc', svm_cv.best_score_)
print('test acc', svm_cv.score(X_test, Y_test))
plot_confusion_matrix(Y_test, svm_cv.predict(X_test), 'SVM')

In [ ]:
tree_cv = GridSearchCV(DecisionTreeClassifier(random_state=2), {'criterion': ['gini', 'entropy'], 'splitter': ['best', 'random'],
    'max_depth': [2*n for n in range(1, 10)], 'max_features': ['sqrt'], 'min_samples_leaf': [1, 2, 4],
    'min_samples_split': [2, 5, 10]}, cv=10).fit(X_train, Y_train)
print('best params', tree_cv.best_params_, 'CV acc', tree_cv.best_score_)
print('test acc', tree_cv.score(X_test, Y_test))
plot_confusion_matrix(Y_test, tree_cv.predict(X_test), 'Decision Tree')

In [ ]:
knn_cv = GridSearchCV(KNeighborsClassifier(), {'n_neighbors': list(range(1, 11)),
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'], 'p': [1, 2]}, cv=10).fit(X_train, Y_train)
print('best params', knn_cv.best_params_, 'CV acc', knn_cv.best_score_)
print('test acc', knn_cv.score(X_test, Y_test))
plot_confusion_matrix(Y_test, knn_cv.predict(X_test), 'KNN')

In [ ]:
results = pd.DataFrame({'CV accuracy': [m.best_score_ for m in [logreg_cv, svm_cv, tree_cv, knn_cv]],
                        'Test accuracy': [m.score(X_test, Y_test) for m in [logreg_cv, svm_cv, tree_cv, knn_cv]]},
                       index=['Logistic Regression', 'SVM', 'Decision Tree', 'KNN'])
results.plot(kind='bar', figsize=(9,4), rot=0, ylim=(0,1)); plt.title('Model accuracy'); plt.show()
results

**Result:** LR, SVM and KNN tie on the 18-sample test set (~83.3%). SVM has the highest cross-validation accuracy (~84.8%), so it is selected as the best model. All models predict every real landing correctly; the errors are false positives.